# Notebook 06 - Train, Test and Evaluate

Goes with **Lesson 08**. Here we learn to measure models **honestly**.

In [ ]:
from pathlib import Path

# The data files are in the "data" folder next to "notebooks".
# In Google Colab, upload the CSV file and it will be found in the current folder.
DATA_DIR = Path("../data") if Path("../data").exists() else Path(".")
print("Reading data from:", DATA_DIR.resolve())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(DATA_DIR / "student_performance.csv")

# Turn text columns into numbers (Lesson 06)
data = df.drop(columns=["student_id"]).copy()
data["internet_access"] = data["internet_access"].map({"yes": 1, "no": 0})
data["extra_classes"] = data["extra_classes"].map({"yes": 1, "no": 0})
data = pd.get_dummies(data, columns=["city_type"], prefix="city", dtype=int)

# Features (X). We remove BOTH answer columns to avoid data leakage.
X = data.drop(columns=["final_score", "result"])
y_class = data["result"]        # classification label: pass / fail
y_number = data["final_score"]  # regression label: a number

print("Features used:", list(X.columns))

## 1. Split into training, validation and test sets (60 / 20 / 20)

We split twice: first keep 20 percent aside as the final **test** set. Then split the rest into **training** and **validation**.

In [ ]:
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(X, y_class, test_size=0.2, random_state=42, stratify=y_class)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

print("Training:  ", len(X_train), "students (the model learns from these)")
print("Validation:", len(X_val), "students (we use these to choose settings)")
print("Test:      ", len(X_test), "students (locked away until the very end)")

`stratify` keeps the same pass/fail share in every part, so no part is unfairly easy.

## 2. Overfitting with a decision tree

A deeper tree can ask more questions. Watch the training and validation accuracy as the tree grows.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier

depths = range(1, 16)
train_scores, val_scores = [], []
for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    train_scores.append(tree.score(X_train, y_train))
    val_scores.append(tree.score(X_val, y_val))

plt.figure(figsize=(8, 4))
plt.plot(depths, train_scores, marker="o", label="training accuracy")
plt.plot(depths, val_scores, marker="o", label="validation accuracy")
plt.xlabel("Tree depth (how many questions it can ask)")
plt.ylabel("Accuracy")
plt.title("Overfitting: training goes to 1.0, validation does not follow")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

best_depth = depths[val_scores.index(max(val_scores))]
print("Best depth on validation data:", best_depth)
print(f"Depth 15 -> training {train_scores[-1]:.2f}, validation {val_scores[-1]:.2f}  <- big gap = overfitting")

- **Left side (depth 1):** both scores lower - the tree is too simple (**underfitting**, high bias).
- **Right side (depth 15):** training is perfect, validation is worse - the tree **memorised** (**overfitting**, high variance).
- The best depth is where the **validation** score is highest.

## 3. Choose the best model using the validation set

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

candidates = {
    f"Decision tree (depth {best_depth})": DecisionTreeClassifier(max_depth=best_depth, random_state=42),
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random forest": RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42),
}
for name, model in candidates.items():
    model.fit(X_train, y_train)
    print(f"{name:28} validation accuracy = {model.score(X_val, y_val):.3f}")

## 4. Cross-validation: a more reliable score

With small data, one validation split can be lucky or unlucky. **5-fold cross-validation** trains 5 times on different parts and averages the results.

In [ ]:
from sklearn.model_selection import cross_val_score

for name, model in candidates.items():
    scores = cross_val_score(model, X_temp, y_temp, cv=5)
    print(f"{name:28} mean = {scores.mean():.3f}   (5 scores: {scores.round(2)})")

Random forest has the best average score. Logistic regression is only a little lower and is much easier to **explain** (teachers can see how much each habit matters). In real projects you weigh accuracy against simplicity. Here we pick **logistic regression**. Try switching `final_model` to the random forest afterwards and compare.

## 5. The final test - used only ONCE

We train the chosen model on training + validation data, then test it on the locked test set.

In [ ]:
final_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
final_model.fit(X_temp, y_temp)
y_pred = final_model.predict(X_test)

print(f"Final test accuracy: {final_model.score(X_test, y_test):.3f}")

## 6. The confusion matrix

Our goal is to find students who may **fail**, so teachers can help them. So **"fail" is the positive class**.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

labels = ["fail", "pass"]   # first = positive class
cm = confusion_matrix(y_test, y_pred, labels=labels)
print(cm)

ConfusionMatrixDisplay(cm, display_labels=labels).plot(cmap="Blues")
plt.title("Rows = real answer, columns = model's answer")
plt.show()

In [ ]:
TP = cm[0, 0]   # really fail, predicted fail
FN = cm[0, 1]   # really fail, predicted pass  (missed!)
FP = cm[1, 0]   # really pass, predicted fail  (false alarm)
TN = cm[1, 1]   # really pass, predicted pass

print("TP =", TP, " FN =", FN, " FP =", FP, " TN =", TN, " total =", TP + FN + FP + TN)

accuracy = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)

print()
print("Calculated BY HAND:")
print(f"accuracy  = {accuracy:.3f}")
print(f"precision = {precision:.3f}   (when it says fail, how often is it right?)")
print(f"recall    = {recall:.3f}   (of all real fails, how many did it catch?)")
print(f"F1-score  = {f1:.3f}")

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

print("Calculated by SCIKIT-LEARN:")
print(f"precision = {precision_score(y_test, y_pred, pos_label='fail'):.3f}")
print(f"recall    = {recall_score(y_test, y_pred, pos_label='fail'):.3f}")
print(f"F1-score  = {f1_score(y_test, y_pred, pos_label='fail'):.3f}")
print()
print(classification_report(y_test, y_pred))

The hand numbers and scikit-learn numbers match. The `classification_report` shows precision, recall and F1 for **both** classes at once. `support` = how many real examples of that class are in the test set.

## 7. Why accuracy alone can fool you

A "lazy model" that always says **pass**:

In [ ]:
lazy_predictions = ["pass"] * len(y_test)
lazy_accuracy = (pd.Series(lazy_predictions, index=y_test.index) == y_test).mean()
lazy_recall = recall_score(y_test, lazy_predictions, pos_label="fail", zero_division=0)
print(f"Lazy model accuracy: {lazy_accuracy:.2f}  <- looks OK!")
print(f"Lazy model recall for 'fail': {lazy_recall:.2f}  <- it catches NO failing student")

## 8. Epochs and loss

Some models learn in rounds called **epochs**. `SGDClassifier` is a logistic-regression-like model trained with gradient descent, one epoch at a time. We record the **loss** after every epoch.

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import log_loss

scaler = StandardScaler().fit(X_train)
Xtr, Xva = scaler.transform(X_train), scaler.transform(X_val)

sgd = SGDClassifier(loss="log_loss", learning_rate="constant", eta0=0.01, random_state=42)
train_loss, val_loss = [], []
for epoch in range(40):
    sgd.partial_fit(Xtr, y_train, classes=["fail", "pass"])   # one pass over the data
    train_loss.append(log_loss(y_train, sgd.predict_proba(Xtr)))
    val_loss.append(log_loss(y_val, sgd.predict_proba(Xva)))

plt.figure(figsize=(8, 4))
plt.plot(range(1, 41), train_loss, label="training loss")
plt.plot(range(1, 41), val_loss, label="validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss (lower is better)")
plt.title("Loss goes down as the model trains for more epochs")
plt.legend()
plt.show()
print(f"Loss after epoch 1: {train_loss[0]:.3f}   after epoch 40: {train_loss[-1]:.3f}")

## 9. Evaluating a regression model (predicting the number)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

Xn_train, Xn_test, yn_train, yn_test = train_test_split(X, y_number, test_size=0.2, random_state=42)
reg = LinearRegression().fit(Xn_train, yn_train)
pred = reg.predict(Xn_test)

print(f"MAE  = {mean_absolute_error(yn_test, pred):.2f} marks  (average mistake)")
print(f"RMSE = {mean_squared_error(yn_test, pred) ** 0.5:.2f} marks  (punishes big mistakes more)")
print(f"R2   = {r2_score(yn_test, pred):.3f}        (1.0 would be perfect)")

plt.figure(figsize=(5, 5))
plt.scatter(yn_test, pred, alpha=0.7)
plt.plot([20, 90], [20, 90], color="red", label="perfect prediction")
plt.xlabel("Real final score")
plt.ylabel("Predicted final score")
plt.legend()
plt.show()

## Summary

```
split -> train -> compare on validation (or cross-validation) -> pick one -> test ONCE -> report precision, recall, F1
```

Go back to **Lesson 08** and do the **Check** section.